In [1]:
import pandas as pd

In [2]:

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
import joblib  # To save the models
import warnings

warnings.filterwarnings('ignore') # Clean up output
print("Libraries Imported Successfully!")

Libraries Imported Successfully!


In [3]:
# 1. Create Dummy User Data (Input for training)
data = {
    'Age': [23, 58, 35, 42, 29, 65, 31, 27],
    'Monthly_Income': [45000, 120000, 85000, 150000, 60000, 40000, 250000, 55000],
    'Risk_Score': [28, 8, 22, 15, 18, 5, 29, 12],
    'Risk_Label': ['Aggressive', 'Conservative', 'Aggressive', 'Moderate', 'Moderate', 'Conservative', 'Aggressive', 'Moderate']
}

df_users = pd.DataFrame(data)

# 2. Separate Inputs (X) and Answers (y)
X = df_users[['Age', 'Monthly_Income', 'Risk_Score']]
y = df_users['Risk_Label']

# 3. Train the Model (Random Forest)
risk_model = RandomForestClassifier(n_estimators=50, random_state=42)
risk_model.fit(X, y)

print("Risk Model Trained Successfully!")
print("Example Prediction for Age 25, Income 50k, Score 28:", risk_model.predict([[25, 50000, 28]]))

Risk Model Trained Successfully!
Example Prediction for Age 25, Income 50k, Score 28: ['Aggressive']


In [4]:
joblib.dump(risk_model, 'risk_model.pkl')
print("File 'risk_model.pkl' saved to your folder.")

File 'risk_model.pkl' saved to your folder.


In [5]:
# 1. Create Stock Data (Features: Annual Return %, Volatility Beta)
# Beta < 1 is Safe, Beta > 1 is Risky
stock_data = {
    'Stock_Name': ['TCS', 'Infosys', 'HDFC Bank', 'ICICI Bank', 'HUL', 'ITC', 'Nestle', 'Reliance', 'L&T',
                   'Tata Motors', 'Adani Ent', 'Zomato', 'Bajaj Finance', 'Nifty50 ETF', 'Gold BeES'],
    'Sector': ['IT', 'IT', 'Bank', 'Bank', 'FMCG', 'FMCG', 'FMCG', 'Oil/Energy', 'Construction', 
               'Auto', 'Metals', 'Tech', 'Finance', 'Index', 'Commodity'],
    'Annual_Return': [12, 11, 14, 18, 10, 15, 9, 13, 20, 35, 40, 45, 22, 12, 8],
    'Volatility_Beta': [0.6, 0.7, 0.9, 1.1, 0.4, 0.5, 0.3, 1.0, 1.2, 1.5, 2.2, 2.5, 1.4, 1.0, 0.2]
}

df_stocks = pd.DataFrame(stock_data)
print("Stock Data Created. First 5 rows:")
print(df_stocks.head())

Stock Data Created. First 5 rows:
   Stock_Name Sector  Annual_Return  Volatility_Beta
0         TCS     IT             12              0.6
1     Infosys     IT             11              0.7
2   HDFC Bank   Bank             14              0.9
3  ICICI Bank   Bank             18              1.1
4         HUL   FMCG             10              0.4


In [6]:
# 1. Select features for clustering (Math only understands numbers)
X_stocks = df_stocks[['Annual_Return', 'Volatility_Beta']]

# 2. Initialize KMeans with 3 Clusters (Safe, Moderate, Risky)
kmeans = KMeans(n_clusters=3, random_state=42)

# 3. Train on the stock data
df_stocks['Cluster_ID'] = kmeans.fit_predict(X_stocks)

print("Clustering Complete. Here are the Cluster Centers (Average values):")
print(kmeans.cluster_centers_)
# Column 0 = Avg Return, Column 1 = Avg Volatility

Clustering Complete. Here are the Cluster Centers (Average values):
[[18.75        1.05      ]
 [40.          2.06666667]
 [11.125       0.6375    ]]


In [7]:
# NOTE: The cluster IDs (0, 1, 2) are random. We must map them manually based on the print output above.
# Usually:
# High Return + High Beta = Risky
# Low Return + Low Beta = Safe
# Medium = Moderate

# Let's create a function to automatically name them to be safe
def name_cluster(row):
    # This logic depends on the specific result of your training, but generally:
    if row['Volatility_Beta'] > 1.4:
        return 'Risky' # High Volatility
    elif row['Volatility_Beta'] < 0.8:
        return 'Safe' # Low Volatility
    else:
        return 'Moderate'

df_stocks['Risk_Category'] = df_stocks.apply(name_cluster, axis=1)

# Save this processed data for the website
df_stocks.to_csv('clustered_stocks.csv', index=False)
print("File 'clustered_stocks.csv' saved. Ready for the Website!")
print(df_stocks[['Stock_Name', 'Cluster_ID', 'Risk_Category']])

File 'clustered_stocks.csv' saved. Ready for the Website!
       Stock_Name  Cluster_ID Risk_Category
0             TCS           2          Safe
1         Infosys           2          Safe
2       HDFC Bank           2      Moderate
3      ICICI Bank           0      Moderate
4             HUL           2          Safe
5             ITC           0          Safe
6          Nestle           2          Safe
7        Reliance           2      Moderate
8             L&T           0      Moderate
9     Tata Motors           1         Risky
10      Adani Ent           1         Risky
11         Zomato           1         Risky
12  Bajaj Finance           0      Moderate
13    Nifty50 ETF           2      Moderate
14      Gold BeES           2          Safe
